# **Ensemble Learning Classification and Regression**

Completed using Iris dataset (Classification) and Diabetes dataset (Regression)

# Classification

## Load the data

In [ ]:
import time
import pandas as pd
from sklearn.datasets import load_iris, load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error
from sklearn.ensemble import (
    VotingClassifier, RandomForestClassifier, BaggingClassifier,
    AdaBoostClassifier, RandomForestRegressor, GradientBoostingRegressor,
    AdaBoostRegressor
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier

# Load Iris dataset
iris = load_iris()
X_class = iris.data
y_class = iris.target


## Data Preprocessing



In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.2, random_state=42)

scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)


## Modeling


### Voting Classifier - Hard voting

In [ ]:
log_clf = LogisticRegression(random_state=42)
rnd_clf = RandomForestClassifier(random_state=42)
svm_clf = SVC(random_state=42)

voting_clf_hard = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf)],
    voting='hard'
)

start_time = time.time()
voting_clf_hard.fit(X_train_c_scaled, y_train_c)
time_hard = time.time() - start_time
print(f"Hard Voting Training Time: {time_hard:.6f} seconds")


Hard Voting Training Time: 0.245905 seconds


### Voting Classifier - Soft voting

In [ ]:
svm_clf_soft = SVC(probability=True, random_state=42)

voting_clf_soft = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf_soft)],
    voting='soft'
)

start_time = time.time()
voting_clf_soft.fit(X_train_c_scaled, y_train_c)
time_soft = time.time() - start_time
print(f"Soft Voting Training Time: {time_soft:.6f} seconds")


Soft Voting Training Time: 0.222240 seconds


### Bagging - Small number of estimators

In [ ]:
bag_clf_small = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), n_estimators=10,
    max_samples=100, bootstrap=True, random_state=42
)

start_time = time.time()
bag_clf_small.fit(X_train_c_scaled, y_train_c)
time_bag_small = time.time() - start_time
print(f"Bagging (Small) Training Time: {time_bag_small:.6f} seconds")


Bagging (Small) Training Time: 0.038645 seconds


### Bagging - Large number of estimators

In [ ]:
bag_clf_large = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), n_estimators=500,
    max_samples=100, bootstrap=True, random_state=42
)

start_time = time.time()
bag_clf_large.fit(X_train_c_scaled, y_train_c)
time_bag_large = time.time() - start_time
print(f"Bagging (Large) Training Time: {time_bag_large:.6f} seconds")


Bagging (Large) Training Time: 1.681727 seconds


### Bagging - Out-of-Bag Evaluation

In [ ]:
bag_clf_oob = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), n_estimators=500,
    bootstrap=True, oob_score=True, random_state=42
)

start_time = time.time()
bag_clf_oob.fit(X_train_c_scaled, y_train_c)
time_bag_oob = time.time() - start_time
print(f"Bagging (OOB) Training Time: {time_bag_oob:.6f} seconds")
print(f"OOB Score: {bag_clf_oob.oob_score_:.4f}")


Bagging (OOB) Training Time: 2.555580 seconds
OOB Score: 0.9500


### Pasting

In [ ]:
pasting_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), n_estimators=100,
    max_samples=100, bootstrap=False, random_state=42
)

start_time = time.time()
pasting_clf.fit(X_train_c_scaled, y_train_c)
time_pasting = time.time() - start_time
print(f"Pasting Training Time: {time_pasting:.6f} seconds")


Pasting Training Time: 0.724574 seconds


### Random Patches

In [ ]:
rp_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), n_estimators=100,
    max_samples=0.75, bootstrap=True, max_features=0.75, bootstrap_features=True, random_state=42
)

start_time = time.time()
rp_clf.fit(X_train_c_scaled, y_train_c)
time_rp = time.time() - start_time
print(f"Random Patches Training Time: {time_rp:.6f} seconds")


Random Patches Training Time: 0.598937 seconds


### Random Subspaces

In [ ]:
rs_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), n_estimators=100,
    max_samples=1.0, bootstrap=False, max_features=0.75, bootstrap_features=True, random_state=42
)

start_time = time.time()
rs_clf.fit(X_train_c_scaled, y_train_c)
time_rs = time.time() - start_time
print(f"Random Subspaces Training Time: {time_rs:.6f} seconds")


Random Subspaces Training Time: 0.610077 seconds


### Random Forest

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)

start_time = time.time()
rf_clf.fit(X_train_c_scaled, y_train_c)
time_rf = time.time() - start_time
print(f"Random Forest Training Time: {time_rf:.6f} seconds")


Random Forest Training Time: 0.423864 seconds


### Random Forest - Hyperparameters tuning


In [ ]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 4, None],
    'min_samples_split': [2, 5]
}

grid_search_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5)

start_time = time.time()
grid_search_rf.fit(X_train_c_scaled, y_train_c)
time_rf_grid = time.time() - start_time

best_rf_model = grid_search_rf.best_estimator_

print(f"RF GridSearch Training Time: {time_rf_grid:.6f} seconds")
print(f"Best RF Parameters: {grid_search_rf.best_params_}")


RF GridSearch Training Time: 37.598261 seconds
Best RF Parameters: {'max_depth': 4, 'min_samples_split': 2, 'n_estimators': 50}


### AdaBoost - small Learning rate

In [ ]:
ada_clf_small = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1, random_state=42), n_estimators=100,
    learning_rate=0.1, random_state=42
)

start_time = time.time()
ada_clf_small.fit(X_train_c_scaled, y_train_c)
time_ada_small = time.time() - start_time
print(f"AdaBoost (Small LR) Training Time: {time_ada_small:.6f} seconds")


AdaBoost (Small LR) Training Time: 0.212883 seconds


### AdaBoost - large Learning rate

In [ ]:
ada_clf_large = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1, random_state=42), n_estimators=100,
    learning_rate=1.0, random_state=42
)

start_time = time.time()
ada_clf_large.fit(X_train_c_scaled, y_train_c)
time_ada_large = time.time() - start_time
print(f"AdaBoost (Large LR) Training Time: {time_ada_large:.6f} seconds")


AdaBoost (Large LR) Training Time: 0.202047 seconds


### Best model metrics

In [ ]:
models_to_evaluate = {
    "Voting (Soft)": voting_clf_soft,
    "Bagging (OOB)": bag_clf_oob,
    "Random Forest (Tuned)": best_rf_model,
    "AdaBoost (Large LR)": ada_clf_large
}

for name, model in models_to_evaluate.items():
    y_pred = model.predict(X_test_c_scaled)
    acc = accuracy_score(y_test_c, y_pred)
    f1 = f1_score(y_test_c, y_pred, average='weighted')
    print(f"{name:25} | Accuracy: {acc:.4f} | F1 Score: {f1:.4f}")


Voting (Soft)             | Accuracy: 1.0000 | F1 Score: 1.0000
Bagging (OOB)             | Accuracy: 1.0000 | F1 Score: 1.0000
Random Forest (Tuned)     | Accuracy: 1.0000 | F1 Score: 1.0000
AdaBoost (Large LR)       | Accuracy: 0.9333 | F1 Score: 0.9333


# Regression

## Load the data

In [ ]:
diabetes = load_diabetes()
X_reg = diabetes.data
y_reg = diabetes.target


## Data Preprocessing

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

scaler_r = StandardScaler()
X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)


## Modeling


### Model 1: Random Forest

In [ ]:
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)

start_time = time.time()
rf_reg.fit(X_train_r_scaled, y_train_r)
time_rf_reg = time.time() - start_time

mse_rf = mean_squared_error(y_test_r, rf_reg.predict(X_test_r_scaled))
print(f"Random Forest Regressor Time: {time_rf_reg:.6f} seconds | MSE: {mse_rf:.2f}")


Random Forest Regressor Time: 0.407030 seconds | MSE: 2959.18


### Model 2: Gradient Boosting

In [ ]:
gb_reg = GradientBoostingRegressor(n_estimators=100, random_state=42)

start_time = time.time()
gb_reg.fit(X_train_r_scaled, y_train_r)
time_gb_reg = time.time() - start_time

mse_gb = mean_squared_error(y_test_r, gb_reg.predict(X_test_r_scaled))
print(f"Gradient Boosting Regressor Time: {time_gb_reg:.6f} seconds | MSE: {mse_gb:.2f}")


Gradient Boosting Regressor Time: 0.191968 seconds | MSE: 2898.20


### Model 3: AdaBoost

In [ ]:
ada_reg = AdaBoostRegressor(n_estimators=100, random_state=42)

start_time = time.time()
ada_reg.fit(X_train_r_scaled, y_train_r)
time_ada_reg = time.time() - start_time

mse_ada = mean_squared_error(y_test_r, ada_reg.predict(X_test_r_scaled))
print(f"AdaBoost Regressor Time: {time_ada_reg:.6f} seconds | MSE: {mse_ada:.2f}")


AdaBoost Regressor Time: 0.201290 seconds | MSE: 2998.45


### Model 4: Support Vector Regressor

In [ ]:
svr_reg = SVR(kernel='linear')

start_time = time.time()
svr_reg.fit(X_train_r_scaled, y_train_r)
time_svr_reg = time.time() - start_time

mse_svr = mean_squared_error(y_test_r, svr_reg.predict(X_test_r_scaled))
print(f"SVR Time: {time_svr_reg:.6f} seconds | MSE: {mse_svr:.2f}")


SVR Time: 0.010609 seconds | MSE: 2939.81


### Model 5: Linear Regression

In [ ]:
lr_reg = LinearRegression()

start_time = time.time()
lr_reg.fit(X_train_r_scaled, y_train_r)
time_lr_reg = time.time() - start_time

mse_lr = mean_squared_error(y_test_r, lr_reg.predict(X_test_r_scaled))
print(f"Linear Regression Time: {time_lr_reg:.6f} seconds | MSE: {mse_lr:.2f}")


Linear Regression Time: 0.030293 seconds | MSE: 2900.19


# Results

### 1. Classification Comparison (Iris Dataset)

**Results Summary Table**

| Model Configuration | Training Time (Sec) | Accuracy | F1 Score |
| :--- | :--- | :--- | :--- |
| Voting Classifier (Hard) | \~0.2459 | - | - |
| **Voting Classifier (Soft)** | \~0.2222 | **1.0000** | **1.0000** |
| Bagging (Small - 10 est.) | \~0.0386 | - | - |
| Bagging (Large - 500 est.) | \~1.6817 | - | - |
| **Bagging (OOB - 500 est.)** | \~2.5555 | **1.0000** | **1.0000** |
| Pasting | \~0.7245 | - | - |
| Random Patches | \~0.5989 | - | - |
| Random Subspaces | \~0.6100 | - | - |
| Random Forest (Default) | \~0.4238 | - | - |
| **Random Forest (GridSearch Tuned)** | \~37.5982 | **1.0000** | **1.0000** |
| AdaBoost (Small LR = 0.1) | \~0.2128 | - | - |
| AdaBoost (Large LR = 1.0) | \~0.2020 | 0.9333 | 0.9333 |

**Justification of Results:**
* **Training Time:** As expected, the number of estimators directly impacts training time. Bagging with 10 estimators took only \~0.038s, while scaling to 500 estimators took \~1.68s. The GridSearch Random Forest was by far the most computationally expensive (\~37.59s) because it trains the ensemble across multiple hyperparameter combinations using 5-fold cross-validation.
* **Performance:** Most ensemble methods achieve perfect accuracy on the Iris dataset because the classes are highly separable. However, AdaBoost with a large learning rate (1.0) saw a drop in accuracy (0.9333), which indicates it may have over-corrected on training errors and slightly overfit the data.
* **Best Model:** **Voting (Soft), Bagging (OOB), and Random Forest (Tuned)** all tie for the best performance with a perfect 1.0 accuracy and F1 score. Considering the tradeoff between training time and performance, **Voting (Soft)** is highly efficient (\~0.22s) compared to the tuned Random Forest.

---

### 2. Regression Comparison (Diabetes Dataset)

**Results Summary Table**

| Regression Model | Training Time (Sec) | Mean Squared Error (MSE) |
| :--- | :--- | :--- |
| Random Forest | \~0.4070 | 2959.18 |
| **Gradient Boosting** | \~0.1919 | **2898.20** |
| AdaBoost | \~0.2012 | 2998.45 |
| Support Vector Regressor (Linear) | \~0.0106 | 2939.81 |
| Linear Regression | \~0.0302 | 2900.19 |

**Justification of Results:**
* **Training Time:** The single algorithms (SVR and Linear Regression) are extremely fast (\~0.01s to \~0.03s). The ensemble methods take an order of magnitude longer (\~0.19s to \~0.40s) because they rely on building 100 sequential or parallel decision trees.
* **Performance:** Linear models perform very well on this dataset, but the ensemble approach of **Gradient Boosting** managed to capture slightly more complex relationships, edging out Linear Regression to achieve the lowest MSE. AdaBoost and Random Forest performed slightly worse, likely struggling to smoothly model the linear trends compared to Gradient Boosting's residual-correction approach.
* **Best Model:** **Gradient Boosting Regressor** is the best model for this dataset. It achieved the lowest MSE (**2898.20**) in a highly reasonable training time (\~0.19s).

---

### 3. Comparison with Previous SVM and Decision Tree Results

**Classification Insights:**
* **Accuracy:** A single Decision Tree and an SVM (Polynomial Kernel, Degree 3) both achieved 1.0000 accuracy. The Ensemble methods (Random Forest, Voting, Bagging) matched this perfect score but did not exceed it, simply because the dataset is already perfectly separable by simpler models.
* **Training Time Justification:** The single Decision Tree (\~0.005s) and SVM are vastly faster than their ensemble counterparts (e.g., Bagging 500 took \~2.5s, Tuned RF took \~37.5s). For a simple dataset like Iris, using complex ensembles introduces unnecessary computational overhead without providing extra predictive value.

**Regression Insights:**
* **Performance (MSE):** The single Decision Tree Regressor performed very poorly (MSE: 3568.97) because standard trees struggle with smooth continuous data. The ensemble methods (Random Forest, AdaBoost, Gradient Boosting) completely fixed this weakness, all scoring under 3000 MSE.
* **Overall Winner:** Gradient Boosting (MSE: 2898.20) slightly outperformed the previous Linear Regression (MSE: 2900.19).
* **Training Time Justification:** While the simple Linear Regression is significantly faster (\~0.02s vs Gradient Boosting's \~0.19s) and offers nearly identical performance, Gradient Boosting proves that combining many weak learners successfully overcomes the severe limitations of a single Decision Tree Regressor.